In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

import os
os.makedirs("outputs", exist_ok=True)

In [2]:
# 1. LOAD DATA

print("=" * 60)
print("  STUDENT LEARNING BEHAVIOUR CLUSTERING")
print("=" * 60)

# Try loading from file, else generate realistic synthetic data
try:
    df = pd.read_csv("data/xAPI-Edu-Data.csv")
    print(f"✓ Loaded real dataset: {df.shape[0]} rows × {df.shape[1]} cols")
except FileNotFoundError:
    print("⚠  Real dataset not found — generating synthetic data (same schema)")
    np.random.seed(42)
    n = 480
    df = pd.DataFrame({
        "gender": np.random.choice(["M", "F"], n),
        "NationalITy": np.random.choice(["Kuwait","Jordan","Palestine","Iraq","Lebanon","Tunis","Saudi","Egypt","Syria","USA","Iran","Lybia","Morocco","Venezuela"], n),
        "PlaceofBirth": np.random.choice(["Kuwait","Jordan","Palestine","Iraq","Lebanon","Tunis","Saudi","Egypt","Syria","USA","Iran","Lybia","Morocco","Venezuela"], n),
        "StageID": np.random.choice(["lowerlevel","MiddleSchool","HighSchool"], n),
        "GradeID": np.random.choice(["G-02","G-04","G-05","G-06","G-07","G-08","G-09","G-10","G-11","G-12"], n),
        "SectionID": np.random.choice(["A","B","C"], n),
        "Topic": np.random.choice(["IT","Math","Arabic","Science","English","Quran","Spanish","French","History","Biology","Chemistry","Geology"], n),
        "Semester": np.random.choice(["First","Second"], n),
        "Relation": np.random.choice(["Father","Mum"], n),
        "raisedhands": np.random.randint(0, 100, n),
        "VisITedResources": np.random.randint(0, 100, n),
        "AnnouncementsView": np.random.randint(0, 100, n),
        "Discussion": np.random.randint(0, 100, n),
        "ParentAnsweringSurvey": np.random.choice(["Yes","No"], n),
        "ParentschoolSatisfaction": np.random.choice(["Good","Bad"], n),
        "StudentAbsenceDays": np.random.choice(["Under-7","Above-7"], n),
        "Class": np.random.choice(["L","M","H"], n, p=[0.3, 0.4, 0.3]),
    })
    print(f"✓ Synthetic dataset created: {df.shape[0]} rows × {df.shape[1]} cols")

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

  STUDENT LEARNING BEHAVIOUR CLUSTERING
⚠  Real dataset not found — generating synthetic data (same schema)
✓ Synthetic dataset created: 480 rows × 17 cols

Dataset shape: (480, 17)
Columns: ['gender', 'NationalITy', 'PlaceofBirth', 'StageID', 'GradeID', 'SectionID', 'Topic', 'Semester', 'Relation', 'raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion', 'ParentAnsweringSurvey', 'ParentschoolSatisfaction', 'StudentAbsenceDays', 'Class']

Missing values:
Series([], dtype: int64)


In [3]:
# 2. PREPROCESSING
# ─────────────────────────────────────────────
print("\n[2] Preprocessing...")

# Encode categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
df_enc = df.copy()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

# Feature selection — numeric + high-value encoded
numeric_features = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']
cat_features     = ['gender', 'Relation', 'ParentAnsweringSurvey', 'ParentschoolSatisfaction', 'StudentAbsenceDays']
features = numeric_features + cat_features

X = df_enc[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"✓ Feature matrix: {X_scaled.shape}")

# PCA
pca2 = PCA(n_components=2, random_state=42)
pca3 = PCA(n_components=3, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)
X_pca3 = pca3.fit_transform(X_scaled)
print(f"✓ PCA 2D variance explained: {pca2.explained_variance_ratio_.sum()*100:.1f}%")
print(f"✓ PCA 3D variance explained: {pca3.explained_variance_ratio_.sum()*100:.1f}%")

# t-SNE
print("  Running t-SNE (this takes ~10s)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# ─────────────────────────────────────────────
# 3. OPTIMAL K  — ELBOW + SILHOUETTE
# ─────────────────────────────────────────────
print("\n[3] Finding optimal number of clusters...")

K_range = range(2, 11)
inertias, sil_scores, db_scores, ch_scores = [], [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

optimal_k = list(K_range)[np.argmax(sil_scores)]
print(f"✓ Optimal K = {optimal_k}  (silhouette = {max(sil_scores):.3f})")


[2] Preprocessing...
✓ Feature matrix: (480, 9)
✓ PCA 2D variance explained: 25.9%
✓ PCA 3D variance explained: 38.0%
  Running t-SNE (this takes ~10s)...

[3] Finding optimal number of clusters...
✓ Optimal K = 4  (silhouette = 0.123)


In [4]:
# 4. CLUSTERING ALGORITHMS
# ─────────────────────────────────────────────
print(f"\n[4] Running 5 clustering algorithms (K={optimal_k})...")

results = {}

# K-Means
km_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
results['KMeans'] = km_model.fit_predict(X_scaled)

# Mini-Batch K-Means
mbkm = MiniBatchKMeans(n_clusters=optimal_k, random_state=42, batch_size=100)
results['MiniBatchKMeans'] = mbkm.fit_predict(X_scaled)

# Agglomerative
agg = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
results['Agglomerative'] = agg.fit_predict(X_scaled)

# DBSCAN (eps tuned automatically)
from sklearn.neighbors import NearestNeighbors
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
eps_auto = np.percentile(distances[:, -1], 90)
db = DBSCAN(eps=eps_auto, min_samples=5)
db_labels = db.fit_predict(X_scaled)
n_noise = (db_labels == -1).sum()
results['DBSCAN'] = db_labels
print(f"  DBSCAN: eps={eps_auto:.3f}, noise={n_noise} points, clusters={len(set(db_labels))-int(-1 in db_labels)}")

# Gaussian Mixture Model
gmm = GaussianMixture(n_components=optimal_k, random_state=42, covariance_type='full')
results['GaussianMixture'] = gmm.fit_predict(X_scaled)

# Evaluation table
print("\n  Algorithm Performance:")
print(f"  {'Algorithm':<20} {'Silhouette':>12} {'Davies-Bouldin':>16} {'Calinski-H':>12}")
print("  " + "-" * 62)
eval_rows = []
for name, labels in results.items():
    valid = labels[labels != -1]
    X_valid = X_scaled[labels != -1]
    if len(set(valid)) < 2:
        continue
    sil = silhouette_score(X_valid, valid)
    db_  = davies_bouldin_score(X_valid, valid)
    ch_  = calinski_harabasz_score(X_valid, valid)
    eval_rows.append({'Algorithm': name, 'Silhouette': sil, 'Davies_Bouldin': db_, 'Calinski_Harabasz': ch_})
    print(f"  {name:<20} {sil:>12.4f} {db_:>16.4f} {ch_:>12.2f}")

best_algo = max(eval_rows, key=lambda x: x['Silhouette'])['Algorithm']
print(f"\n  ★ Best algorithm: {best_algo}")


[4] Running 5 clustering algorithms (K=4)...
  DBSCAN: eps=2.327, noise=5 points, clusters=1

  Algorithm Performance:
  Algorithm              Silhouette   Davies-Bouldin   Calinski-H
  --------------------------------------------------------------
  KMeans                     0.1230           2.5394        47.11
  MiniBatchKMeans            0.0830           2.6979        38.50
  Agglomerative              0.0786           3.1768        35.01
  GaussianMixture            0.0846           2.4899        39.67

  ★ Best algorithm: KMeans


In [5]:
# 5. CLUSTER PROFILING

print("\n[5] Profiling clusters...")

df['Cluster'] = results['KMeans']
cluster_profile = df.groupby('Cluster')[numeric_features].mean().round(2)
print("\n  Cluster Profiles (mean values):")
print(cluster_profile.to_string())

# Persona naming based on engagement level
engagement = cluster_profile.mean(axis=1)
cluster_names = {}
sorted_clusters = engagement.sort_values().index.tolist()
personas = ["Disengaged Learner", "Passive Observer", "Average Student", "Active Participant", "High Achiever"]
for i, c in enumerate(sorted_clusters):
    cluster_names[c] = personas[min(i, len(personas)-1)]

df['Persona'] = df['Cluster'].map(cluster_names)
print(f"\n  Cluster personas: {cluster_names}")


[5] Profiling clusters...

  Cluster Profiles (mean values):
         raisedhands  VisITedResources  AnnouncementsView  Discussion
Cluster                                                              
0              50.09             51.58              52.40       53.83
1              51.64             50.16              46.97       51.79
2              48.18             53.50              50.75       51.49
3              51.93             44.64              47.48       49.45

  Cluster personas: {3: 'Disengaged Learner', 1: 'Passive Observer', 2: 'Average Student', 0: 'Active Participant'}


In [6]:
# 6. VISUALIZATIONS
# ─────────────────────────────────────────────
print("\n[6] Generating visualizations...")

COLORS = ['#534AB7', '#1D9E75', '#D85A30', '#185FA5', '#639922', '#D4537E', '#BA7517']
palette = {c: COLORS[i % len(COLORS)] for i, c in enumerate(sorted(df['Cluster'].unique()))}

# ── Fig 1: Elbow + Silhouette ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Optimal K Selection", fontsize=14, fontweight='bold', y=1.02)

axes[0].plot(list(K_range), inertias, 'o-', color='#534AB7', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method (Inertia)'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].axvline(optimal_k, color='#D85A30', linestyle='--', label=f'Optimal K={optimal_k}')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range), sil_scores, 's-', color='#1D9E75', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
axes[1].axvline(optimal_k, color='#D85A30', linestyle='--'); axes[1].grid(alpha=0.3)

axes[2].plot(list(K_range), db_scores, 'D-', color='#D85A30', linewidth=2, markersize=7)
axes[2].set_title('Davies-Bouldin Score'); axes[2].set_xlabel('K'); axes[2].set_ylabel('Score (lower=better)')
axes[2].axvline(optimal_k, color='#534AB7', linestyle='--'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/01_optimal_k.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/01_optimal_k.png")

# ── Fig 2: PCA 2D scatter grid ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Clustering Results — PCA 2D Projection", fontsize=14, fontweight='bold')
algo_names = list(results.keys())
for idx, (name, labels) in enumerate(results.items()):
    ax = axes[idx // 3][idx % 3]
    unique = sorted(set(labels))
    for cl in unique:
        mask = labels == cl
        color = '#AAAAAA' if cl == -1 else COLORS[cl % len(COLORS)]
        label = 'Noise' if cl == -1 else f'Cluster {cl}'
        ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=color, label=label, alpha=0.6, s=25, edgecolors='none')
    ax.set_title(name); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8, markerscale=1.2); ax.grid(alpha=0.2)

# 6th panel: t-SNE
ax = axes[1][2]
for cl in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cl
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=COLORS[cl % len(COLORS)], label=cluster_names[cl], alpha=0.6, s=25)
ax.set_title('t-SNE (K-Means labels)'); ax.legend(fontsize=7); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("outputs/02_clustering_comparison.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/02_clustering_comparison.png")

# ── Fig 3: Dendrogram ──────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
sample_idx = np.random.choice(len(X_scaled), min(200, len(X_scaled)), replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90,
           color_threshold=0.7*max(Z[:,2]))
ax.set_title("Hierarchical Clustering Dendrogram (Ward linkage — 200 samples)", fontsize=13)
ax.set_xlabel("Sample index"); ax.set_ylabel("Distance")
ax.axhline(y=np.percentile(Z[:,2], 70), color='#D85A30', linestyle='--', label='Cut threshold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/03_dendrogram.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/03_dendrogram.png")

# ── Fig 4: Feature correlation heatmap ────────
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_enc[features + ['Class']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Heatmap", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/04_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/04_correlation_heatmap.png")

# ── Fig 5: Radar chart per cluster ────────────
from matplotlib.patches import FancyArrowPatch
angles = np.linspace(0, 2*np.pi, len(numeric_features), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, optimal_k, figsize=(4*optimal_k, 4), subplot_kw=dict(polar=True))
fig.suptitle("Cluster Engagement Radar Charts", fontsize=13, fontweight='bold')
if optimal_k == 1: axes = [axes]

norm_profile = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min() + 1e-9)
for cl_idx, (cl, row) in enumerate(norm_profile.iterrows()):
    vals = row.values.tolist() + row.values[:1].tolist()
    ax = axes[cl_idx]
    ax.plot(angles, vals, color=COLORS[cl % len(COLORS)], linewidth=2)
    ax.fill(angles, vals, color=COLORS[cl % len(COLORS)], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Raised\nHands', 'Resources', 'Announcements', 'Discussion'], fontsize=8)
    ax.set_title(f"C{cl}: {cluster_names.get(cl, 'Cluster ' + str(cl))}", fontsize=9, pad=12)
    ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/05_radar_charts.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/05_radar_charts.png")

# ── Fig 6: Algorithm comparison bar ───────────
eval_df = pd.DataFrame(eval_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Algorithm Evaluation Metrics", fontsize=13, fontweight='bold')

colors_bar = [COLORS[i % len(COLORS)] for i in range(len(eval_df))]
axes[0].bar(eval_df['Algorithm'], eval_df['Silhouette'], color=colors_bar, edgecolor='white', linewidth=0.5)
axes[0].set_title('Silhouette Score (higher = better)'); axes[0].set_ylabel('Score')
axes[0].tick_params(axis='x', rotation=20); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(eval_df['Algorithm'], eval_df['Davies_Bouldin'], color=colors_bar, edgecolor='white', linewidth=0.5)
axes[1].set_title('Davies-Bouldin Score (lower = better)'); axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=20); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/06_algorithm_comparison.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/06_algorithm_comparison.png")

# ── Fig 7: Cluster distribution ───────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Cluster Distribution Analysis", fontsize=13, fontweight='bold')

cluster_counts = df['Cluster'].value_counts().sort_index()
wedge_colors = [COLORS[i % len(COLORS)] for i in cluster_counts.index]
axes[0].pie(cluster_counts, labels=[cluster_names.get(c, f'C{c}') for c in cluster_counts.index],
            autopct='%1.1f%%', colors=wedge_colors, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title("Cluster Size Distribution")

df_melt = df[numeric_features + ['Cluster']].melt(id_vars='Cluster', var_name='Feature', value_name='Value')
sns.boxplot(data=df_melt, x='Feature', y='Value', hue='Cluster', ax=axes[1],
            palette={c: COLORS[c % len(COLORS)] for c in df['Cluster'].unique()})
axes[1].set_title("Feature Distribution by Cluster")
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(title='Cluster', fontsize=8)

plt.tight_layout()
plt.savefig("outputs/07_cluster_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/07_cluster_distribution.png")

# ── Fig 8: PCA 3D ─────────────────────────────
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
for cl in sorted(df['Cluster'].unique()):
    mask = df['Cluster'].values == cl
    ax.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
               c=COLORS[cl % len(COLORS)], label=cluster_names.get(cl, f'C{cl}'),
               alpha=0.6, s=30)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')
ax.set_title("3D PCA Projection — K-Means Clusters", fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("outputs/08_pca_3d.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/08_pca_3d.png")

# ── Fig 9: Summary dashboard ──────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Student Learning Behaviour Clustering — Summary Dashboard",
             fontsize=16, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# KMeans PCA scatter
ax1 = fig.add_subplot(gs[0, 0])
for cl in sorted(df['Cluster'].unique()):
    mask = df['Cluster'].values == cl
    ax1.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=COLORS[cl % len(COLORS)],
                label=cluster_names.get(cl,''), alpha=0.6, s=20)
ax1.set_title('K-Means (PCA 2D)'); ax1.legend(fontsize=7); ax1.grid(alpha=0.2)

# Silhouette bar
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(eval_df['Algorithm'], eval_df['Silhouette'],
        color=[COLORS[i%len(COLORS)] for i in range(len(eval_df))], edgecolor='white')
ax2.set_title('Silhouette Scores'); ax2.tick_params(axis='x', rotation=25); ax2.grid(axis='y', alpha=0.3)

# Cluster size pie
ax3 = fig.add_subplot(gs[0, 2])
ax3.pie(cluster_counts, labels=[f'C{c}' for c in cluster_counts.index],
        autopct='%1.0f%%', colors=wedge_colors, startangle=140,
        wedgeprops=dict(edgecolor='white', linewidth=1.2))
ax3.set_title('Cluster Sizes')

# Feature boxplot
ax4 = fig.add_subplot(gs[1, :2])
sns.boxplot(data=df_melt, x='Feature', y='Value', hue='Cluster', ax=ax4,
            palette={c: COLORS[c % len(COLORS)] for c in df['Cluster'].unique()})
ax4.set_title('Feature Distribution by Cluster'); ax4.grid(axis='y', alpha=0.3)
ax4.legend(title='Cluster', fontsize=8, loc='upper right')

# Cluster profile heatmap
ax5 = fig.add_subplot(gs[1, 2])
cp_norm = (cluster_profile.T - cluster_profile.T.min()) / (cluster_profile.T.max() - cluster_profile.T.min() + 1e-9)
sns.heatmap(cp_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax5,
            linewidths=0.5, cbar=False)
ax5.set_title('Normalized Cluster Profile')
ax5.set_xticklabels([f'C{c}' for c in cluster_profile.index])

plt.savefig("outputs/09_dashboard.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ outputs/09_dashboard.png")


[6] Generating visualizations...
  ✓ outputs/01_optimal_k.png
  ✓ outputs/02_clustering_comparison.png
  ✓ outputs/03_dendrogram.png
  ✓ outputs/04_correlation_heatmap.png
  ✓ outputs/05_radar_charts.png
  ✓ outputs/06_algorithm_comparison.png
  ✓ outputs/07_cluster_distribution.png
  ✓ outputs/08_pca_3d.png
  ✓ outputs/09_dashboard.png


In [7]:
# 7. SAVE RESULTS

df.to_csv("outputs/clustered_students.csv", index=False)
cluster_profile.to_csv("outputs/cluster_profiles.csv")
eval_df.to_csv("outputs/algorithm_evaluation.csv", index=False)

print("\n" + "=" * 60)
print("  ALL DONE!")
print("=" * 60)
print(f"\n  Dataset       : {df.shape[0]} students × {df.shape[1]} features")
print(f"  Optimal K     : {optimal_k}")
print(f"  Best algorithm: {best_algo}")
print(f"  Best Silhouette: {max(sil_scores):.4f}")
print(f"\n  Personas identified:")
for cl, name in sorted(cluster_names.items()):
    count = (df['Cluster'] == cl).sum()
    print(f"    Cluster {cl}: {name}  ({count} students, {count/len(df)*100:.1f}%)")
print(f"\n  Outputs saved to: outputs/")
print("=" * 60)


  ALL DONE!

  Dataset       : 480 students × 19 features
  Optimal K     : 4
  Best algorithm: KMeans
  Best Silhouette: 0.1230

  Personas identified:
    Cluster 0: Active Participant  (104 students, 21.7%)
    Cluster 1: Passive Observer  (116 students, 24.2%)
    Cluster 2: Average Student  (130 students, 27.1%)
    Cluster 3: Disengaged Learner  (130 students, 27.1%)

  Outputs saved to: outputs/
